# 3.0.0 - Phân loại trạng thái thị trường theo Walk-Forward Expanding Quantile

### 🎯 Mục tiêu & Nguyên tắc chống Data Leakage:
- **Tránh rò rỉ dữ liệu (No Look-Ahead Bias):** Tuyệt đối **không** tính phân vị trên toàn bộ tập dữ liệu 1990–2026 trước khi chia Train/Test.
- **Cơ chế Walk-Forward / Expanding Window:** Tại mỗi thời điểm $t$, ngưỡng phân vị chỉ được tính toán dựa trên dữ liệu lịch sử từ quá khứ đến thời điểm đó ($\le t$).

### 📐 Công thức toán học:
Tại mỗi thời điểm dự báo $t$:
$$Q_{33, t} = \text{Quantile}(\text{Volatility}_{20D, \le t}, 0.33)$$
$$Q_{67, t} = \text{Quantile}(\text{Volatility}_{20D, \le t}, 0.67)$$

Sau đó xác định trạng thái thị trường ($\text{Volatility\_Regime}_t$):
$$\text{Volatility\_Regime}_t = \begin{cases} \text{Low}, & \text{Volatility}_{20D, t} \le Q_{33, t} \\ \text{Normal}, & Q_{33, t} < \text{Volatility}_{20D, t} \le Q_{67, t} \\ \text{High}, & \text{Volatility}_{20D, t} > Q_{67, t} \end{cases}$$

---
- **Dữ liệu đầu vào:** `dataset/2.2.0_feature.csv`
- **Dữ liệu đầu ra:** `dataset/3.0.0_regime.csv` (thêm 1 cột duy nhất: `Volatility_Regime`)

In [ ]:
import os
import numpy as np
import pandas as pd

# 1. Nạp tập dữ liệu phiên bản 2.2.0_feature.csv
dataset_dir = 'dataset' if os.path.exists('dataset') else ('../dataset' if os.path.exists('../dataset') else '.')
input_path = os.path.join(dataset_dir, '2.2.0_feature.csv')
print(f"Đọc dữ liệu từ: {input_path}")
df = pd.read_csv(input_path)

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
print(f"Kích thước dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột.")
display(df.head(3))

In [ ]:
# 2. Tính toán Q33(t) và Q67(t) theo cơ chế Walk-Forward / Expanding Window
# Thiết lập cửa sổ khởi tạo tối thiểu min_periods = 252 (1 năm giao dịch lịch sử) để phân phối có ý nghĩa thống kê
min_periods = 252

# Tại mỗi ngày t, chỉ tính phân vị trên dữ liệu từ quá khứ đến ngày t (<= t)
q33_t = df['Volatility_20D'].expanding(min_periods=min_periods).quantile(0.33).bfill()
q67_t = df['Volatility_20D'].expanding(min_periods=min_periods).quantile(0.67).bfill()

print("Đã tính toán xong ngưỡng động Q33(t) và Q67(t) không bị leak thông tin tương lai.")
print(f"- Ngưỡng tại ngày đầu có đủ 1 năm ({df['Date'].iloc[min_periods].date()}): Q33 = {q33_t.iloc[min_periods]:.6f}, Q67 = {q67_t.iloc[min_periods]:.6f}")
print(f"- Ngưỡng tại ngày kết thúc ({df['Date'].iloc[-1].date()}): Q33 = {q33_t.iloc[-1]:.6f}, Q67 = {q67_t.iloc[-1]:.6f}")

In [ ]:
# 3. Phân loại Volatility_Regime (Low / Normal / High)
conditions = [
    df['Volatility_20D'] <= q33_t,
    (df['Volatility_20D'] > q33_t) & (df['Volatility_20D'] <= q67_t),
    df['Volatility_20D'] > q67_t
]
choices = ['Low', 'Normal', 'High']

df['Volatility_Regime'] = np.select(conditions, choices, default=np.nan)

print("=== PHÂN PHỐI SỐ LƯỢNG PHIÊN GIAO DỊCH THEO VOLATILITY_REGIME ===")
counts = df['Volatility_Regime'].value_counts(dropna=False)
display(counts)

print("\nTỷ lệ phần trăm (%):\n", df['Volatility_Regime'].value_counts(normalize=True, dropna=True) * 100)

In [ ]:
# 4. Xuất file kết quả phiên bản 3.0.0_regime.csv
output_path = os.path.join(dataset_dir, '3.0.0_regime.csv')
df.to_csv(output_path, index=False)

print(f"✅ ĐÃ XUẤT THÀNH CÔNG TẠI: {output_path}")
print(f"Kích thước Dataset mới: {df.shape[0]} dòng, {df.shape[1]} cột (thêm cột Volatility_Regime).")
display(df[['Date', 'Close', 'Return_1D', 'Volatility_20D', 'Volatility_Regime']].dropna().sample(5, random_state=42))